In [1]:
# Must be FIRST - before any other imports
import os
os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="

import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  if Path("/content/drive").exists():
    print("Nothing to initialize, driver is already mounted!")
  else:
    from google.colab import drive
    drive.mount('/content/drive')
    colab_dir = f"/content/drive/MyDrive/colab"
    if colab_dir not in sys.path:
      sys.path.insert(0, colab_dir)
    from colab_setup import init
    paths = init("singular-learning-theory")
    print("Installing tex")
    !apt-get install texlive-fonts-recommended texlive-latex-extra texlive-pictures dvipng

  datadir = "/content/drive/MyDrive/colab/outputs/singular-learning-theory/mixtures/binom2d"
else:
  from pathlib import Path
  paths = {'repo': Path.cwd(), 'output': Path.cwd() / 'outputs'}
  paths['output'].mkdir(exist_ok=True)
  print("📍 Running locally")
  from pathlib import Path

  ROOT = Path.cwd().parents[1]
  basedir = f"{ROOT}/outputs/mixture/binom2d"
  datadir = f"{basedir}/data"
  outputdir = Path(f"{basedir}/posterior")


if not outputdir.exists():
  outputdir.mkdir(exist_ok=True)
  print(f"Created {outputdir}!")

print(f"Using datadir={datadir}")
print(f"Using outputdir={outputdir}")

📍 Running locally
Using datadir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/data
Using outputdir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/posterior


In [2]:
import pandas as pd

dgps_file = f"{datadir}/dgp.csv"
dgps=pd.read_csv(dgps_file, index_col=0).reset_index()
dgps.head()

,dsid,p0,p1,w0,w1
0,regular,0.15,0.35,0.75,0.25
1,e-singular,0.25,0.35,0.75,0.25
2,singular1,0.15,0.35,1.00,0.00


In [3]:
def find_truth_by_dsid(dsid: str):
    dgp = dgps.query(f"dsid=='{dsid}'")
    truth = dgp[["p0", "p1", "w0", "w1"]].iloc[0].tolist()
    return truth

In [4]:
import pytensor
import pymc as pm
import numpy as np

n_components = 2
n_trials = 100
dsid="singular1"
regime = 250
probs = [0.15, 0.35]

dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")
X = dataset.iloc[:, 0].to_numpy()
# Mixture of 2 Poisson variables
with pm.Model() as model:
  w = pm.Dirichlet("w", a=np.array([1, 1]))  # 2 mixture weights

  components = []
  for i in range(n_components):
    components.append(pm.Binomial.dist(n=100, p=probs[i]))

  like = pm.Mixture("like", w=w, comp_dists=components, observed=X)
  idata = pm.sample(nuts_sampler="numpyro")  # or "blackjax"

idata

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Inference data with groups:
	> posterior
	> sample_stats
	> observed_data